## PromptTemplates

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

In [10]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url=BASE_URL, 
    api_key=API_KEY, 
    model='gpt-5.1',
    temperature=1.8
)

In [30]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

### Static Messages

In [4]:
prompt = f'tell me a joke on dogs.'
result = model.invoke(prompt)
print(result.content)

Why did the dog sit in the shade?

Because he didn’t want to be a hot dog. 🌭🐶


### Prompt Using F-strings

In [7]:
prompt = 'Tell me a joke on {topic}'
result = model.invoke(prompt.format(topic='cats'))
print(result.content)

Why don’t cats play poker in the jungle?

Too many cheetahs. 🐾😼


In [ ]:
topic = 'lion'
prompt = f'Tell me a joke on {topic}'

print(f'F-string prompt content: {prompt}\n')

result = model.invoke(prompt)
print(result.content)

F-string prompt content: Tell me a joke on lion
Why don’t lions like fast food?

Because they can’t catch it. 🦁🍔


### Prompt Template

In [16]:
from langchain_core.prompts import PromptTemplate

pt = PromptTemplate(
    template = 'One line roaster on {topic}'
)

pt

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='One line roaster on {topic}')

In [21]:
prompt = pt.invoke('Lannisters')
prompt

StringPromptValue(text='One line roaster on Lannisters')

In [20]:
result = model.invoke(prompt)
result.content

'Gold can’t buy honor, but it did a great job renting out the Lannisters’ dignity.'

In [22]:
# Using chaining
chain = pt | model
result = chain.invoke('Joffery Lannister')
result

AIMessage(content='Even Joffrey’s own crossbow seemed embarrassed to be seen with him.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 17, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 34, 'engine_ttlt_ms': 246, 'pre_inference_ms': 101, 'service_tbt_ms': 9, 'service_ttft_ms': 609, 'service_ttlt_ms': 832, 'total_duration_ms': 735, 'user_visible_ttft_ms': 507}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-Dco6yZNbSWUnx20pZ1CF7O7upkUub', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e017d-80a2-75b3-a4c8-b916dd51a247-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_toke

In [23]:
type(result)

langchain_core.messages.ai.AIMessage

### ChatPromptTemplate

This is better than Prompt Template 

Two ways to invoke 
1. from template - similar to PromptTemplate
2. form messages - list of messages (default)

This is inherirted from the above calss

In [28]:
from langchain_core.prompts import ChatPromptTemplate

template = 'Two liner roast on {topic}'
pt = ChatPromptTemplate.from_template(template)

pt

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Two liner roast on {topic}'), additional_kwargs={})])

In [34]:
# pt = ChatPromptTemplate(
#     HumanMessage('Two liner roast on {topic}')
# ) -> Depriciated

pt = ChatPromptTemplate(
    ('human', 'Two liner roast on {topic}')
)
pt

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='human'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Two liner roast on {topic}'), additional_kwargs={})])

In [36]:
prompt = pt.invoke('Sansa Stark')
prompt

ChatPromptValue(messages=[HumanMessage(content='human', additional_kwargs={}, response_metadata={}), HumanMessage(content='Two liner roast on Sansa Stark', additional_kwargs={}, response_metadata={})])

In [40]:
chain = pt | model

result = chain.invoke('Sansa Stark')
print(result.content)

You’ve survived so much, Sansa, yet somehow your personality is still stuck in Season 1.  
You played the game of thrones and technically won, but only because everyone more interesting died first.


#### Multiple Messages

In [ ]:
messages = [
    ('system', 'You are a professional commedian, where you roast the {topic} in about 2 lines each'),
    ('human', 'I like to get some jokes on {character}, give me {count} of it')
]

pt = ChatPromptTemplate.from_messages(messages)
pt

ChatPromptTemplate(input_variables=['character', 'count', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='You are a professional commedian, where you roast the {topic} in about 2 lines each'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'count'], input_types={}, partial_variables={}, template='I like to get some jokes on {character}, give me {count} of it'), additional_kwargs={})])

In [56]:
prompt = pt.invoke({'topic': 'fictional character', 'character': 'jon snow', 'count': 3})
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional commedian, where you roast the fictional character in about 2 lines each', additional_kwargs={}, response_metadata={}), HumanMessage(content='I like to get some jokes on jon snow, give me 3 of it', additional_kwargs={}, response_metadata={})])

In [57]:
result = model.invoke(prompt)
print(result.content)

1. Jon Snow spent eight seasons proving he “knows nothing,” then topped it off by not even recognizing his own aunt…on a dragon.  
2. Only Jon Snow could rise from the dead and still not manage a personality upgrade.  
3. The man’s entire leadership style is just heavy breathing, long pauses, and then doing exactly the worst possible thing with honor.


In [68]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

messages = [
    SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], template='You are a professional commedian, where you roast the {topic}')),
    HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'count'], template='I like to get some jokes on {character}, give me {count} of it'))
]

pt = ChatPromptTemplate.from_messages(messages)
pt

ChatPromptTemplate(input_variables=['character', 'count', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='You are a professional commedian, where you roast the {topic}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'count'], input_types={}, partial_variables={}, template='I like to get some jokes on {character}, give me {count} of it'), additional_kwargs={})])

In [70]:
chain = pt | model
result = chain.invoke({'topic': 'fictional character', 'character': 'Ned Stark', 'count': 3})
print(result.content)

1. Ned Stark’s life motto was “Winter is Coming,” but he forgot the second part: “and so is common sense—maybe listen to your wife once in a while.”

2. Ned Stark doing politics is like bringing a practice sword to a dragon fight: honorable, wooden, and absolutely getting melted.

3. Ned’s strategy in King’s Landing was basically:  
   “Step 1: Tell the truth.  
    Step 2: Trust everyone.  
    Step 3: Be surprised when you die in episode 9.”
